# Computer Graphics — CO1

**2310218L.CO.1** — *Apply OpenGL graphics primitives to develop graphics applications.*
**[L3]**

---

## The idea

A screen is a grid of dots. Nothing on it is a line or a box — those are ideas we impose.
A **primitive** is the smallest shape a system can draw, and every picture is built from
them.

```
picture  ->  primitives  ->  pixels  ->  screen
```

Ours are the rectangle, line, circle and text.

![the four primitives, composed into the diagram](diagrams/cg-primitives.png)

---

## Everything goes through one surface

```cpp
class Canvas {
private:
    int    width_;
    int    height_;
    Pixel* buffer_;
public:
    void  clear(Pixel p = PX_EMPTY);
    void  setPixel(int x, int y, Pixel p);
    Pixel getPixel(int x, int y) const;
    bool  inBounds(int x, int y) const;

    void drawLineBresenham(int x0, int y0, int x1, int y1, Pixel p);
    void drawCircleBresenham(int cx, int cy, int radius, Pixel p);
    void drawBox(int x, int y, int w, int h, const std::string& title);
    void drawText(int x, int y, const std::string& text);
    void render(std::ostream& os) const;
};
```
<sub>src/ui/Canvas.h — declarations</sub>

```cpp
void Canvas::setPixel(int x, int y, Pixel p) {
    if (inBounds(x, y)) buffer_[y * width_ + x] = p;
}
```
<sub>src/ui/Canvas.cpp:41</sub>

**One place writes a pixel.** Swapping the console for a graphics window means rewriting
`render()` — and nothing above it.

---

## Rectangle and text

```cpp
void Canvas::drawBox(int x, int y, int w, int h, const std::string& title) {
    if (w < 2 || h < 2) return;

    for (int i = x + 1; i < x + w - 1; ++i) {
        setPixel(i, y,         '-');
        setPixel(i, y + h - 1, '-');
    }
    for (int j = y + 1; j < y + h - 1; ++j) {
        setPixel(x,         j, '|');
        setPixel(x + w - 1, j, '|');
    }
    setPixel(x,         y,         '+');
    setPixel(x + w - 1, y,         '+');
    setPixel(x,         y + h - 1, '+');
    setPixel(x + w - 1, y + h - 1, '+');

    if (!title.empty() && w > 4) {
        std::string t = title;
        if (static_cast<int>(t.size()) > w - 4) t = t.substr(0, w - 4);
        drawText(x + 2, y, t);
    }
}

void Canvas::drawText(int x, int y, const std::string& text) {
    for (size_t i = 0; i < text.size(); ++i)
        setPixel(x + static_cast<int>(i), y, text[i]);
}
```
<sub>src/ui/Canvas.cpp:331</sub>

Even the rectangle is a composition — four runs of lines plus corners.

---

## Composing them into the application

```cpp
c.drawBox(1,  1, 24, 6, "REGISTER FILE");
c.drawBox(29, 1, 20, 6, "CONTROL UNIT");
c.drawBox(52, 1, 22, 6, "MEMORY");
c.drawBox(20, 13, 24, 7, "ALU");
```
<sub>src/ui/ConsoleView.cpp:66</sub>

```cpp
Pixel busPix = sig.busActive ? PX_ACTIVE : PX_WIRE;
const int busY = 10;
c.drawLineBresenham(2, busY, W - 3, busY, busPix);
c.drawText(2, busY - 1, "DATA BUS");

Pixel regPix = (sig.regRead || sig.regWrite) ? PX_ACTIVE : PX_WIRE;
Pixel memPix = (sig.memRead || sig.memWrite) ? PX_ACTIVE : PX_WIRE;
Pixel aluPix = sig.aluEnable                 ? PX_ACTIVE : PX_WIRE;

c.drawLineBresenham(12, 7, 12, busY - 1, regPix);
c.drawLineBresenham(63, 7, 63, busY - 1, memPix);
c.drawLineBresenham(31, busY + 1, 31, 12,  aluPix);

c.setPixel(12, busY, regPix == PX_ACTIVE ? 'O' : 'o');
c.setPixel(63, busY, memPix == PX_ACTIVE ? 'O' : 'o');
```
<sub>src/ui/ConsoleView.cpp:63 — condensed</sub>

**The picture is redrawn from machine state every step.** An active wire and an idle wire
are the *same call* with a different pixel — which is what makes the datapath light up.

---

## Onto the screen

```cpp
void Canvas::render(std::ostream& os) const {
    for (int y = 0; y < height_; ++y) {
        int last = -1;
        for (int x = 0; x < width_; ++x)
            if (buffer_[y * width_ + x] != PX_EMPTY) last = x;
        for (int x = 0; x <= last; ++x)
            os << buffer_[y * width_ + x];
        os << '\n';
    }
}
```
<sub>src/ui/Canvas.cpp:354</sub>

---

## In one minute

1. **A screen is only dots** — a line is something we impose.
2. Four primitives: rectangle, line, circle, text.
3. **All of them go through `setPixel`** — one point of contact with memory.
4. The application is the composition: boxes, a bus, wires, junctions.
5. **It's live** — redrawn from machine state each step, so the same call renders an idle
   wire or an active one.